# DACN Eval Matrix — Multi-model × Multi-fixture × Ablation

**Runtime**: GPU T4 (Runtime > Change runtime type > T4 GPU)

**Pipeline**: Install Ollama → Pull models → Clone repo → Run eval matrix → Aggregate results

**Eval dimensions**:
- **Models**: qwen2.5:3b, qwen2.5:7b, gemma4:e2b (configurable)
- **Fixtures**: IDOR, SSRF, SQLi (3 challenge types)
- **Ablation configs**: baseline, C2 (anti-loop), C3 (KG RAG), C1 (mid-thinking), all

**Metrics collected**: status, steps, tool_calls, wall_time_s, loop_detected, watchdog_trips, validated_findings, budget (tokens, simulated cost)

**Output**: `eval_results.csv` + `eval_results.json` + summary tables

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# Cell 2: Install Ollama + start server
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print('Ollama server started, PID:', proc.pid)

In [ ]:
# Cell 3: CONFIGURATION — edit this cell before running
# ====================================================================

MODELS = [
    'qwen2.5:3b',   # 1.9 GB — nhanh nhat, test truoc
    # 'qwen2.5:7b', # 4.7 GB — uncomment khi chay full
    # 'gemma4:e2b',  # 7.2 GB — uncomment khi chay full
]

FIXTURES = [
    'data/fixtures/challenge_idor_01',
    'data/fixtures/challenge_ssrf_01',
    'data/fixtures/challenge_sqli_01',
]

ABLATION_CONFIGS = {
    'baseline':  {'enable_anti_loop': False, 'enable_kg': False, 'enable_mid_thinking': False},
    'C2_only':   {'enable_anti_loop': True,  'enable_kg': False, 'enable_mid_thinking': False},
    'C3_only':   {'enable_anti_loop': False, 'enable_kg': True,  'enable_mid_thinking': False},
    'C1_only':   {'enable_anti_loop': False, 'enable_kg': False, 'enable_mid_thinking': True},
    'all':       {'enable_anti_loop': True,  'enable_kg': True,  'enable_mid_thinking': True},
}

MAX_STEPS = 15

total_runs = len(MODELS) * len(FIXTURES) * len(ABLATION_CONFIGS)
print(f'Eval matrix: {len(MODELS)} models x {len(FIXTURES)} fixtures x {len(ABLATION_CONFIGS)} configs = {total_runs} runs')

In [ ]:
# Cell 4: Pull all models
for model in MODELS:
    print(f'\n--- Pulling {model} ---')
    !ollama pull {model}

print('\n--- Available models ---')
!ollama list

In [ ]:
# Cell 5: Clone repo + install deps
!git clone https://github.com/NoSpaceAvailable/DACN.git /content/DACN 2>/dev/null || (cd /content/DACN && git pull)
%cd /content/DACN
!pip install -e . -q
!pip install z3-solver -q
print('\nInstalled OK')

In [ ]:
# Cell 6: Smoke test — verify each model responds
import requests

for model in MODELS:
    r = requests.post('http://localhost:11434/api/generate', json={
        'model': model,
        'prompt': 'Reply with the single word OK.',
        'stream': False,
        'options': {'temperature': 0}
    })
    data = r.json()
    tok_s = data.get('eval_count', 0) / max(1, data.get('eval_duration', 1)) * 1e9
    print(f'{model:20s} -> {data["response"][:50]:30s}  ({tok_s:.1f} tok/s)')

In [ ]:
# Cell 7: Run eval matrix
import os, json, time
from pathlib import Path

os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'

from vapt_orchestrator_safe.engine.dispatcher_runner import DispatcherRunner
from vapt_orchestrator_safe.llm.backend_factory import build_chat_model

results = []
run_idx = 0
total = len(MODELS) * len(FIXTURES) * len(ABLATION_CONFIGS)

for model in MODELS:
    backend_spec = f'ollama:{model}'
    os.environ['LLM_BACKEND_DISPATCHER'] = backend_spec

    for fixture_path in FIXTURES:
        fixture = Path(fixture_path)
        fixture_name = fixture.name

        for config_name, config_flags in ABLATION_CONFIGS.items():
            run_idx += 1
            print(f'\n[{run_idx}/{total}] model={model}  fixture={fixture_name}  config={config_name}')

            chat_model = build_chat_model(backend_spec)

            runner = DispatcherRunner(
                outputs_root=Path(f'outputs/eval_{model.replace(":", "_")}'),
                backend=f'ollama:{model}',
                chat_model=chat_model,
                chat_model_backend_spec=backend_spec,
                max_steps=MAX_STEPS,
                enable_sandbox=False,
                **config_flags,
            )

            t0 = time.perf_counter()
            try:
                summary = runner.run_fixture(fixture)
                wall_s = time.perf_counter() - t0
                status = summary['status']
            except Exception as exc:
                wall_s = time.perf_counter() - t0
                summary = {}
                status = f'error:{type(exc).__name__}'
                print(f'  ERROR: {exc}')

            # Extract metrics from memory.json
            out_dir = summary.get('output_dir', '')
            loop_detected = 0
            if out_dir:
                mem_path = Path(out_dir) / 'memory.json'
                if mem_path.exists():
                    events = json.loads(mem_path.read_text()).get('events', [])
                    loop_detected = sum(1 for e in events if e.get('message') == 'loop_detected')

            row = {
                'model': model,
                'fixture': fixture_name,
                'config': config_name,
                'status': status,
                'stop_reason': summary.get('stop_reason', '?'),
                'steps': summary.get('steps', 0),
                'tool_calls': len(summary.get('tool_invocations', [])),
                'validated_findings': len(summary.get('validated_findings', [])),
                'loop_detected': loop_detected,
                'watchdog_trips': len(summary.get('watchdog_trips', [])),
                'wall_s': round(wall_s, 2),
                'budget_tokens': summary.get('budget', {}).get('simulated_tokens', 0),
                'budget_cost': summary.get('budget', {}).get('simulated_cost', 0),
            }
            results.append(row)

            flag = 'V' if row['validated_findings'] > 0 else '-'
            print(f'  [{flag}] {status}  steps={row["steps"]}  tools={row["tool_calls"]}  '
                  f'findings={row["validated_findings"]}  wall={wall_s:.1f}s')

print(f'\n{"="*60}')
print(f'Eval complete: {len(results)} runs')

In [ ]:
# Cell 8: Build DataFrame + Detection Rate table
import pandas as pd

df = pd.DataFrame(results)
df['detected'] = df['status'].isin(['validated', 'supported']).astype(int)

print('='*70)
print('TABLE 1: Detection Rate by Model (across all configs)')
print('='*70)
t1 = df.groupby('model').agg(
    runs=('detected', 'count'),
    detected=('detected', 'sum'),
    detection_rate=('detected', 'mean'),
    avg_steps=('steps', 'mean'),
    avg_tools=('tool_calls', 'mean'),
    avg_wall_s=('wall_s', 'mean'),
).round(3)
print(t1.to_string())

print(f'\n{"="*70}')
print('TABLE 2: Detection Rate by Model x Fixture')
print('='*70)
t2 = df.pivot_table(
    index='model', columns='fixture', values='detected',
    aggfunc='mean', margins=True, margins_name='Overall',
).round(3)
print(t2.to_string())

print(f'\n{"="*70}')
print('TABLE 3: Detection Rate by Model x Config (ablation)')
print('='*70)
t3 = df.pivot_table(
    index='model', columns='config', values='detected',
    aggfunc='mean', margins=True, margins_name='Overall',
).round(3)
col_order = [c for c in ['baseline', 'C2_only', 'C3_only', 'C1_only', 'all', 'Overall'] if c in t3.columns]
print(t3[col_order].to_string())

In [ ]:
# Cell 9: Ablation contribution analysis (C1/C2/C3 delta vs baseline)
print('='*70)
print('TABLE 4: Ablation — contribution of each novel component')
print('='*70)

baseline_rate = df[df['config']=='baseline'].groupby('model')['detected'].mean()

for cfg in ['C2_only', 'C3_only', 'C1_only', 'all']:
    cfg_rate = df[df['config']==cfg].groupby('model')['detected'].mean()
    delta = (cfg_rate - baseline_rate).round(3)
    print(f'\n  {cfg} vs baseline (detection rate delta):')
    for model, d in delta.items():
        sign = '+' if d >= 0 else ''
        print(f'    {model}: {sign}{d}')

print(f'\n{"="*70}')
print('TABLE 5: Loop & Watchdog events by config')
print('='*70)
t5 = df.groupby('config').agg(
    total_loops=('loop_detected', 'sum'),
    total_watchdogs=('watchdog_trips', 'sum'),
    avg_steps=('steps', 'mean'),
    avg_tools=('tool_calls', 'mean'),
).round(2)
row_order = [c for c in ['baseline', 'C2_only', 'C3_only', 'C1_only', 'all'] if c in t5.index]
print(t5.loc[row_order].to_string())

print(f'\n{"="*70}')
print('TABLE 6: Full results (all runs)')
print('='*70)
display_cols = ['model', 'fixture', 'config', 'status', 'steps', 'tool_calls',
                'validated_findings', 'loop_detected', 'watchdog_trips', 'wall_s']
print(df[display_cols].to_string(index=False))

In [ ]:
# Cell 10: Export results
import json
from datetime import datetime

export = {
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'models': MODELS,
    'fixtures': FIXTURES,
    'configs': list(ABLATION_CONFIGS.keys()),
    'max_steps': MAX_STEPS,
    'total_runs': len(results),
    'runs': results,
    'summary': {
        'by_model': df.groupby('model')['detected'].mean().round(3).to_dict(),
        'by_config': df.groupby('config')['detected'].mean().round(3).to_dict(),
        'by_fixture': df.groupby('fixture')['detected'].mean().round(3).to_dict(),
    },
}

json_path = 'outputs/eval_results.json'
csv_path = 'outputs/eval_results.csv'
os.makedirs('outputs', exist_ok=True)

with open(json_path, 'w') as f:
    json.dump(export, f, indent=2)
df.to_csv(csv_path, index=False)

print(f'Exported:')
print(f'  JSON: {json_path} ({os.path.getsize(json_path)} bytes)')
print(f'  CSV:  {csv_path} ({os.path.getsize(csv_path)} bytes)')
print(f'\nDownload: Files > outputs/')

In [ ]:
# Cell 11 (optional): Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Detection rate by model
ax = axes[0]
rates = df.groupby('model')['detected'].mean()
rates.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Detection Rate by Model')
ax.set_ylabel('Rate')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.8, color='red', linestyle='--', label='Target (80%)')
ax.legend()
ax.tick_params(axis='x', rotation=30)

# Chart 2: Ablation config comparison
ax = axes[1]
config_order = ['baseline', 'C2_only', 'C3_only', 'C1_only', 'all']
cfg_rates = df.groupby('config')['detected'].mean().reindex(config_order)
colors = ['#cccccc', '#4daf4a', '#377eb8', '#ff7f00', '#e41a1c']
cfg_rates.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('Detection Rate by Ablation Config')
ax.set_ylabel('Rate')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.8, color='red', linestyle='--', label='Target (80%)')
ax.legend()
ax.tick_params(axis='x', rotation=30)

# Chart 3: Wall time by config
ax = axes[2]
cfg_wall = df.groupby('config')['wall_s'].mean().reindex(config_order)
cfg_wall.plot(kind='bar', ax=ax, color='#984ea3', edgecolor='black')
ax.set_title('Avg Wall Time by Config')
ax.set_ylabel('Seconds')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('outputs/eval_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/eval_charts.png')